# Notebook 07: HistGradientBoosting Model
## ML-Based Liquidity and Trading-Risk Intelligence for Indian Equities

**Phase:** Phase 8 Batch 4 (Stages 8.9–8.11)
**Target:** High-Risk Event (`hre_5d_train`, 5-day forward horizon)
**Evaluation Protocol:** 10-Fold Expanding-Window Walk-Forward Validation with 5-Session Purging

### Objectives
1. Train `HistGradientBoostingClassifier` across 10 expanding-window walk-forward folds.
2. Utilize all 36 approved candidate features (including F51 `relative_volume`).
3. Leverage native histogram binning for high computational speed and low memory footprint (<500 MB).
4. Exploit native NaN routing during tree splitting.
5. Compare out-of-fold performance (PR-AUC, ROC-AUC, Brier score, calibration) against Logistic Regression and Random Forest.


In [1]:
# Imports and Environment Setup
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add repository root to path regardless of working directory
current_dir = Path('.').resolve()
if (current_dir / 'data' / 'processed').exists():
    REPO_ROOT = current_dir
elif (current_dir.parent / 'data' / 'processed').exists():
    REPO_ROOT = current_dir.parent
else:
    REPO_ROOT = current_dir
    for p in [current_dir] + list(current_dir.parents):
        if (p / 'data' / 'processed').exists():
            REPO_ROOT = p
            break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.modeling.preprocessing import build_hgbt_pipeline, get_model_features
from src.modeling.evaluation import ModelEvaluator

print(f'Repository root: {REPO_ROOT}')
print('Environment initialized successfully.')


Repository root: C:\Users\vighn\Downloads\Indian_Equities_Liquidity_Risk_ML-main\Indian_Equities_Liquidity_Risk_ML-main
Environment initialized successfully.


### 1. HGBT Architecture & 36-Feature Specification

HistGradientBoosting operates directly on raw feature matrices without scaling or imputation.
Missing values are assigned to dedicated histogram bins during construction.


In [2]:
hgbt_features = get_model_features('hist_gradient_boosting')
print(f'HGBT Features ({len(hgbt_features)}):\n')
print(', '.join(hgbt_features))

pipe_hgbt = build_hgbt_pipeline()
print('\nHGBT Pipeline:', pipe_hgbt)


HGBT Features (36):

daily_return, return_5d, return_20d, return_z20, abs_return, price_rel_ma20, vol_5d, vol_20d, vol_ratio_5_20, downside_vol_20d, max_drawdown_20d, rsi_14, vol_of_vol, log_volume, volume_z20, amihud_ratio, amihud_ma20, amihud_z20, log_turnover, turnover_ratio, low_volume_streak, daily_range, daily_range_ma20, range_z20, overnight_gap, overnight_gap_abs, cs_spread_est, cs_spread_ma10, excess_return_1d, rolling_beta_60d, rolling_corr_60d, relative_volume, nifty_vol_20d, nifty_return_20d, vol_beta, vix_close

HGBT Pipeline: Pipeline(steps=[('classifier',
                 HistGradientBoostingClassifier(class_weight='balanced',
                                                l2_regularization=0.1,
                                                learning_rate=0.05,
                                                max_iter=200,
                                                random_state=42))])


### 2. Walk-Forward Validation Across 10 Folds

Load the 10-fold walk-forward validation results for HistGradientBoosting.


In [3]:
metrics_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'supervised_fold_metrics.csv'
fold_metrics = pd.read_csv(metrics_path)

hgbt_metrics = fold_metrics[fold_metrics['model_name'] == 'hist_gradient_boosting']
display_cols = [
    'fold', 'val_start', 'val_end', 'val_rows', 'val_prev',
    'pr_auc', 'roc_auc', 'brier_score', 'f1_50', 'calibrated_threshold', 'f1_calib', 'fit_time_sec'
]
hgbt_metrics[display_cols].sort_values('fold')


,fold,val_start,val_end,val_rows,val_prev,pr_auc,roc_auc,brier_score,f1_50,calibrated_threshold,f1_calib,fit_time_sec
4,0,2023-05-10,2023-08-02,15780,0.186122,0.650597,0.873069,0.113074,0.590103,0.46,0.587078,13.171164
9,1,2023-08-03,2023-10-31,15821,0.187077,0.601890,0.863157,0.140679,0.571429,0.42,0.555453,24.616093
14,2,2023-11-01,2024-01-30,15939,0.195079,0.627140,0.859359,0.119352,0.595831,0.62,0.586433,13.621328
19,3,2024-01-31,2024-04-30,16157,0.178883,0.613290,0.852786,0.143408,0.538205,0.52,0.544809,23.926433
24,4,2024-05-02,2024-07-29,16260,0.170541,0.592373,0.853728,0.111504,0.569595,0.68,0.502045,26.734204
29,5,2024-07-30,2024-10-23,16358,0.193606,0.638130,0.856317,0.117251,0.578750,0.50,0.578750,14.661388
34,6,2024-10-24,2025-01-20,16435,0.177609,0.618625,0.858395,0.118663,0.569030,0.50,0.569030,13.768176
39,7,2025-01-21,2025-04-21,16685,0.179143,0.595775,0.849091,0.167427,0.514038,0.54,0.526635,17.914156
44,8,2025-04-22,2025-07-15,16911,0.177577,0.584358,0.836544,0.111054,0.532460,0.70,0.389746,19.635791
49,9,2025-07-16,2025-10-10,17011,0.184351,0.584362,0.845872,0.123901,0.562491,0.40,0.548628,22.308934


### 3. Comprehensive Model Comparison Leaderboard

Compare all 5 supervised configurations ranked by Mean Out-of-Fold PR-AUC.


In [4]:
comp_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'supervised_model_comparison.csv'
comp_df = pd.read_csv(comp_path)

leaderboard_cols = [
    'model_name', 'mean_pr_auc', 'std_pr_auc', 'min_pr_auc', 'pooled_pr_auc',
    'mean_roc_auc', 'mean_brier_score', 'mean_f1_50', 'mean_f1_calib',
    'mean_calib_threshold', 'total_fit_time_sec'
]
comp_df[leaderboard_cols]


,model_name,mean_pr_auc,std_pr_auc,min_pr_auc,pooled_pr_auc,mean_roc_auc,mean_brier_score,mean_f1_50,mean_f1_calib,mean_calib_threshold,total_fit_time_sec
0,hist_gradient_boosting,0.610654,0.022871,0.584358,0.592605,0.854832,0.126631,0.562193,0.538861,0.534,190.357665
1,random_forest_candidate_b,0.582892,0.035178,0.529108,0.561034,0.836072,0.131976,0.537450,0.525200,0.508,790.045664
2,random_forest_candidate_a,0.581417,0.037692,0.524303,0.559061,0.834624,0.132852,0.533908,0.520586,0.504,682.002488
3,logistic_regression_standard,0.498995,0.048742,0.416555,0.479688,0.826452,0.138233,0.522714,0.515200,0.460,33.695080
4,logistic_regression_robust,0.498994,0.048713,0.416527,0.479676,0.826429,0.138187,0.522696,0.515659,0.458,199.291015


### 4. Market Capitalization Stratification

Evaluate HGBT performance across Large Cap, Mid Cap, and Small Cap segments.


In [5]:
preds_path = REPO_ROOT / 'data' / 'interim' / 'phase8_batch4' / 'supervised_predictions.parquet'
preds_df = pd.read_parquet(preds_path)

sub_df = preds_df.rename(columns={'y_true': 'hre_5d_train', 'prob_hist_gradient_boosting': 'hre_prob'})
cap_eval = ModelEvaluator.evaluate_by_cap_segment(
    sub_df, segment_col='cap_segment', target_col='hre_5d_train', prob_col='hre_prob', threshold=0.50
)
print('HGBT Market Cap Stratification:')
print(cap_eval.to_string(index=False))


HGBT Market Cap Stratification:
cap_segment   pr_auc  roc_auc  threshold       f1  precision   recall  balanced_accuracy  brier_score  log_loss    tp    fp     tn    fn  n_obs  n_pos  prevalence
      large 0.788097 0.938142        0.5 0.440680   0.922654 0.289469           0.642019     0.093816  0.291009  3018   253  46332  7408  57011  10426    0.182877
        mid 0.735477 0.919154        0.5 0.664140   0.557307 0.821646           0.836248     0.110930  0.354428 12268  9745  55592  2663  80268  14931    0.186014
      small 0.630354 0.884433        0.5 0.448538   0.292111 0.965655           0.737166     0.247167  0.690450  4358 10561  10934   155  26008   4513    0.173524
        ALL 0.592605 0.849174        0.5 0.560672   0.488620 0.657650           0.751777     0.126655  0.385807 19644 20559 112858 10226 163287  29870    0.182929
